## Carregando os dados prontos para o banco de dados

#### 1. Importando dependências

In [23]:
import os
import pandas as pd
import psycopg2
from io import StringIO
from dotenv import load_dotenv

#### 2. Carregando variáveis de ambiente

In [24]:

load_dotenv()

USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASSWORD')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

#### 3. Criando conexão com o banco de dados

In [25]:
conn = psycopg2.connect(
    dbname=f'{DB_NAME}',
    user=f'{USER}',
    password=f'{PASSWORD}',
    host=f'{HOST}',
    port=f'{PORT}'
)
cur = conn.cursor()

#### 4. Carregando dados para um DataFrame

In [26]:
df = pd.read_parquet('../data/final/glp-consolidado.parquet')

#### 5. Corrigindo os nomes das colunas

In [27]:
colunas_mapeadas = {
    'Regiao - Sigla': 'regiao',
    'Estado - Sigla': 'estado',
    'Municipio': 'municipio',
    'Revenda': 'revenda',
    'CNPJ da Revenda': 'cnpj',
    'Nome da Rua': 'logradouro',
    'Numero Rua': 'numero',
    'Complemento': 'complemento',
    'Bairro': 'bairro',
    'Cep': 'cep',
    'Produto': 'produto',
    'Data da Coleta': 'data_coleta',
    'Valor de Venda': 'valor_venda',
    'Unidade de Medida': 'unidade_medida',
    'Bandeira': 'bandeira'
}

df.rename(columns=colunas_mapeadas, inplace=True)

#### 6. Criando buffer CSV em memória

In [28]:
buffer = StringIO()
df.to_csv(buffer, encoding='utf-8', index=False, header=False)
buffer.seek(0)

0

#### 7. Copiando dados para a tabela no banco

In [29]:
colunas = ','.join(df.columns)

cur.copy_expert(
    
    f"""
    COPY hist_glp.glp_precos ({colunas}) 
    FROM STDIN 
    WITH (
        FORMAT CSV,
        DELIMITER ',',
        NULL ''
    )
    """,
    buffer
        
)

conn.commit()
cur.close()
conn.close()